# Project 10: Banking Transaction Pattern Analysis

**NAME: Krishan Pratap**

**Adm. No.: 2026B01010801**

**GitHub Link:** https://github.com/CrishPycodes/Career-Shaper-HCL-.git


**Program:** Five-Day Engineering Student Training (First Year – First Semester)  
**Industry:** Banking / FinTech  
**Environment:** Google Colab (Python 3)  
**Libraries Used:** Strictly `numpy` and `pandas` (no other libraries)  
**Input Dataset:** `10_bank_transactions_1200.csv` (1,200-entry expanded synthetic dataset)  
**Cleaned Output:** `10_bank_transactions_1200_cleaned_removed.csv` (1,180 clean records after removing missing rows)  

> **How to Run in Google Colab:** Click the **Files** folder icon on the left sidebar, upload `10_bank_transactions_1200.csv`, then click **Runtime → Run all**.  
> **Disclaimer:** This dataset is synthetic and must not be used for fraud claims. `Flagged` is a supplied label, not proof of fraud, and correlation does not prove causation.

## FR-01: Load and Inspect Transaction Records
Load `10_bank_transactions_1200.csv` using Pandas and inspect shape, date range, column data types, missing values, and sample rows.

In [1]:
import numpy as np
import pandas as pd

# Clean display formatting for Pandas tables
pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", 10)
pd.set_option("display.width", 120)


def load_and_inspect_data(filepath):
    """FR-01: Load CSV transaction records, parse dates, and inspect schema."""
    df = pd.read_csv(filepath)
    df["Transaction_Date"] = pd.to_datetime(df["Transaction_Date"])

    print("=== FR-01: Dataset Inspection ===")
    print(f"Dataset Shape: {df.shape[0]} rows x {df.shape[1]} columns")
    print(
        f"Date Range:    {df['Transaction_Date'].min().date()} to "
        f"{df['Transaction_Date'].max().date()} "
        f"({df['Transaction_Date'].nunique()} unique dates)"
    )
    print("\nColumn Data Types:")
    print(df.dtypes)
    print("\nMissing Values per Column:")
    print(df.isnull().sum())
    print("\nFirst 5 Rows:")
    print(df.head().to_string(index=False))
    return df


FILEPATH = "10_bank_transactions_1200.csv"
df_raw = load_and_inspect_data(FILEPATH)

=== FR-01: Dataset Inspection ===
Dataset Shape: 1200 rows x 8 columns
Date Range:    2025-01-01 to 2025-12-31 (354 unique dates)

Column Data Types:
Transaction_ID              object
Account_Type                object
Transaction_Date    datetime64[ns]
Transaction_Type            object
Amount_INR                 float64
Channel                     object
City                        object
Flagged                     object
dtype: object

Missing Values per Column:
Transaction_ID       0
Account_Type         0
Transaction_Date     0
Transaction_Type     0
Amount_INR          20
Channel              0
City                 0
Flagged              0
dtype: int64

First 5 Rows:
Transaction_ID Account_Type Transaction_Date Transaction_Type  Amount_INR Channel    City Flagged
      TX000000      Savings       2025-02-15            Debit      139.73    Card    Pune      No
      TX000001      Current       2025-01-16           Credit      362.84  Branch    Pune      No
      TX000002      Sa

## FR-02 & FR-03: Remove Missing-Amount Rows, Verify Unique IDs & Export Cleaned CSV
* Completely remove all rows with missing `Amount_INR` values using `.dropna(subset=["Amount_INR"])` instead of replacing them with estimated values.
* Verify 100% primary-key uniqueness of `Transaction_ID`.
* Export the cleaned dataset to `10_bank_transactions_1200_cleaned_removed.csv`.

In [2]:
def clean_transaction_data(df_raw, output_csv="10_bank_transactions_1200_cleaned_removed.csv"):
    """FR-02 & FR-03: Remove rows with missing Amount_INR, verify unique IDs, and save cleaned CSV."""
    df = df_raw.copy()

    # FR-02: Detect and completely remove rows with missing Amount_INR
    rows_before = len(df)
    missing_before = int(df["Amount_INR"].isnull().sum())
    df = df.dropna(subset=["Amount_INR"]).reset_index(drop=True)
    rows_after = len(df)

    print("=== FR-02: Missing Value Removal ===")
    print(f"Rows before cleaning:               {rows_before}")
    print(f"Missing Amount_INR rows removed:    {missing_before} ({missing_before / rows_before * 100:.2f}%)")
    print(f"Rows remaining after cleaning:      {rows_after}")
    print(f"Missing Amount_INR after cleaning:  {int(df['Amount_INR'].isnull().sum())}")

    # FR-03: Check duplicate Transaction_ID values
    duplicate_id_count = int(df["Transaction_ID"].duplicated().sum())
    print("\n=== FR-03: Duplicate Transaction_ID Check ===")
    print(f"Duplicate Transaction_ID count:     {duplicate_id_count}")
    if duplicate_id_count > 0:
        df = df.drop_duplicates(subset=["Transaction_ID"], keep="first").reset_index(drop=True)
        print("Duplicate Transaction_ID records removed.")
    else:
        print("All Transaction_ID values are unique (100% primary-key uniqueness verified).")

    # Export cleaned CSV with uniform YYYY-MM-DD dates
    df_export = df.copy()
    df_export["Transaction_Date"] = df_export["Transaction_Date"].dt.strftime("%Y-%m-%d")
    df_export.to_csv(output_csv, index=False)
    print(f"\nCleaned dataset exported to: {output_csv} ({len(df_export)} rows x {df_export.shape[1]} columns)")

    return df


df_clean = clean_transaction_data(df_raw)

=== FR-02: Missing Value Removal ===
Rows before cleaning:               1200
Missing Amount_INR rows removed:    20 (1.67%)
Rows remaining after cleaning:      1180
Missing Amount_INR after cleaning:  0

=== FR-03: Duplicate Transaction_ID Check ===
Duplicate Transaction_ID count:     0
All Transaction_ID values are unique (100% primary-key uniqueness verified).

Cleaned dataset exported to: 10_bank_transactions_1200_cleaned_removed.csv (1180 rows x 8 columns)


## FR-04 & FR-05: Credit vs. Debit Totals & Core NumPy Statistics
Calculate total Credits and Debits via Pandas on the 1,180 cleaned records, then compute mean, median, and standard deviation of `Amount_INR` using NumPy.

In [3]:
def compute_core_statistics(df):
    """FR-04 & FR-05: Compute Credit/Debit totals (Pandas) and Amount_INR stats (NumPy)."""
    total_value = df["Amount_INR"].sum()

    # FR-04: Calculate total Credits and total Debits
    credit_debit = df.groupby("Transaction_Type")["Amount_INR"].agg(
        Transaction_Count="count",
        Total_Amount_INR="sum"
    )
    credit_debit["Value_Share_Pct"] = (credit_debit["Total_Amount_INR"] / total_value) * 100

    print("=== FR-04: Credit vs. Debit Totals ===")
    print(credit_debit)
    print(f"\nTotal Cleaned Transaction Value (N={len(df)}): INR {total_value:,.2f}")

    # FR-05: Calculate mean, median, and standard deviation using NumPy
    amounts_np = df["Amount_INR"].to_numpy()
    mean_amt = np.mean(amounts_np)
    median_amt = np.median(amounts_np)
    std_amt = np.std(amounts_np, ddof=1)

    print("\n=== FR-05: Transaction Amount Statistics (NumPy) ===")
    print(f"Mean:               INR {mean_amt:,.2f}")
    print(f"Median:             INR {median_amt:,.2f}")
    print(f"Standard Deviation: INR {std_amt:,.2f}")
    print(f"Min / Max:          INR {np.min(amounts_np):,.2f} / INR {np.max(amounts_np):,.2f}")

    return amounts_np


amounts_np = compute_core_statistics(df_clean)

=== FR-04: Credit vs. Debit Totals ===
                  Transaction_Count  Total_Amount_INR  Value_Share_Pct
Transaction_Type                                                      
Credit                          611        376,487.55            51.73
Debit                           569        351,249.28            48.27

Total Cleaned Transaction Value (N=1180): INR 727,736.83

=== FR-05: Transaction Amount Statistics (NumPy) ===
Mean:               INR 616.73
Median:             INR 379.88
Standard Deviation: INR 791.97
Min / Max:          INR 25.28 / INR 6,978.50


## FR-06 & FR-07: Channel Volume & City Transaction Value Comparisons
Compare transaction counts across all 5 channels and total monetary value across all 5 cities.

In [4]:
def compare_channels_and_cities(df):
    """FR-06 & FR-07: Compare transaction volume by Channel and total value by City."""
    total_value = df["Amount_INR"].sum()

    # FR-06: Compare transaction volume and share across channels
    channel_summary = (
        df.groupby("Channel")
        .agg(
            Transaction_Count=("Transaction_ID", "count"),
            Total_Value_INR=("Amount_INR", "sum")
        )
        .sort_values(by="Transaction_Count", ascending=False)
    )
    channel_summary["Volume_Share_Pct"] = (channel_summary["Transaction_Count"] / len(df)) * 100

    print("=== FR-06: Transaction Volume by Channel ===")
    print(channel_summary)

    # FR-07: Compare total transaction value and share across cities
    city_summary = (
        df.groupby("City")
        .agg(
            Total_Value_INR=("Amount_INR", "sum"),
            Transaction_Count=("Transaction_ID", "count")
        )
        .sort_values(by="Total_Value_INR", ascending=False)
    )
    city_summary["Value_Share_Pct"] = (city_summary["Total_Value_INR"] / total_value) * 100

    print("\n=== FR-07: Total Transaction Value by City ===")
    print(city_summary)


compare_channels_and_cities(df_clean)

=== FR-06: Transaction Volume by Channel ===
            Transaction_Count  Total_Value_INR  Volume_Share_Pct
Channel                                                         
NetBanking                257       170,575.90             21.78
Branch                    252       150,099.98             21.36
UPI                       234       144,907.59             19.83
ATM                       233       150,890.35             19.75
Card                      204       111,263.01             17.29

=== FR-07: Total Transaction Value by City ===
         Total_Value_INR  Transaction_Count  Value_Share_Pct
City                                                        
Kolkata       169,020.60                247            23.23
Mumbai        168,652.56                259            23.17
Delhi         144,663.09                250            19.88
Pune          128,863.90                217            17.71
Chennai       116,536.68                207            16.01


## FR-08 & FR-09: Overall & Channel-Level Flagged Record Analysis
Calculate overall flagged count and percentage, then compare flagged percentages across all 5 channels using `pd.crosstab()`.

In [5]:
def analyze_flagged_records(df):
    """FR-08 & FR-09: Calculate overall and channel-level flagged counts and percentages."""
    # FR-08: Overall flagged transaction count and percentage
    flagged_count = int((df["Flagged"] == "Yes").sum())
    flagged_pct = (flagged_count / len(df)) * 100

    print("=== FR-08: Overall Flagged Transaction Summary ===")
    print(f"Flagged Transactions: {flagged_count} out of {len(df)} ({flagged_pct:.2f}%)")
    print("Note: 'Flagged' is a supplied label in a synthetic dataset, not proof of fraud.")

    # FR-09: Compare flagged percentages across transaction channels
    flagged_by_channel = pd.crosstab(df["Channel"], df["Flagged"])
    flagged_by_channel["Total"] = flagged_by_channel.sum(axis=1)
    flagged_by_channel["Flagged_Pct"] = (flagged_by_channel["Yes"] / flagged_by_channel["Total"]) * 100
    flagged_by_channel = flagged_by_channel.sort_values(by="Flagged_Pct", ascending=False)

    print("\n=== FR-09: Flagged Percentage by Channel ===")
    print(flagged_by_channel)


analyze_flagged_records(df_clean)

=== FR-08: Overall Flagged Transaction Summary ===
Flagged Transactions: 50 out of 1180 (4.24%)
Note: 'Flagged' is a supplied label in a synthetic dataset, not proof of fraud.

=== FR-09: Flagged Percentage by Channel ===
Flagged      No  Yes  Total  Flagged_Pct
Channel                                 
Branch      238   14    252         5.56
UPI         223   11    234         4.70
ATM         223   10    233         4.29
NetBanking  248    9    257         3.50
Card        198    6    204         2.94


## FR-10 & FR-11: Top 10 Largest Transactions & Account-Type Totals
Extract the 10 largest transactions by `Amount_INR` and group Credit and Debit totals by `Account_Type`.

In [6]:
def analyze_top_and_accounts(df):
    """FR-10 & FR-11: Display 10 largest transactions and Credit/Debit totals by Account_Type."""
    # FR-10: Find and display the 10 largest transactions
    top_10 = df.nlargest(10, "Amount_INR")[
        [
            "Transaction_ID", "Transaction_Date", "Account_Type",
            "Transaction_Type", "Amount_INR", "Channel", "City", "Flagged"
        ]
    ]
    print("=== FR-10: Top 10 Largest Transactions ===")
    print(top_10.to_string(index=False))

    # FR-11: Group Credit and Debit totals by Account_Type
    account_summary = df.groupby(["Account_Type", "Transaction_Type"])["Amount_INR"].agg(
        Transaction_Count="count",
        Total_Amount_INR="sum"
    )
    print("\n=== FR-11: Credit and Debit Totals by Account Type ===")
    print(account_summary)


analyze_top_and_accounts(df_clean)

=== FR-10: Top 10 Largest Transactions ===
Transaction_ID Transaction_Date Account_Type Transaction_Type  Amount_INR    Channel    City Flagged
      TX000200       2025-04-20      Student            Debit    6,978.50        UPI    Pune      No
      TX000679       2025-10-25      Student           Credit    6,950.00        UPI   Delhi      No
      TX000158       2025-05-07      Current           Credit    6,919.54        ATM    Pune      No
      TX000165       2025-04-04      Student           Credit    6,511.37        ATM  Mumbai      No
      TX000244       2025-04-21      Savings            Debit    6,400.10 NetBanking Kolkata      No
      TX001107       2025-09-28      Student           Credit    6,114.32 NetBanking   Delhi      No
      TX000799       2025-08-01      Current            Debit    5,932.25 NetBanking   Delhi      No
      TX001119       2025-07-08      Student            Debit    5,847.11        ATM    Pune      No
      TX000171       2025-02-28      Savings    

## FR-12 & FR-13: Ordered Amount Bands & NumPy Boolean Indexing (> INR 1,000)
* Segment transactions into 5 ordered monetary bands using `pd.cut()`.
* Isolate transactions above INR 1,000 using NumPy Boolean indexing.

In [7]:
def segment_bands_and_threshold(df, amounts_np, threshold=1000.0):
    """FR-12 & FR-13: Segment amounts into 5 ordered bands and filter > threshold via NumPy."""
    # FR-12: Create 5 ordered transaction amount bands
    bins = [0, 250, 500, 1000, 2500, np.inf]
    labels = [
        "Very Low (<=250)",
        "Low (250-500)",
        "Medium (500-1,000)",
        "High (1,000-2,500)",
        "Very High (>2,500)"
    ]
    df["Amount_Band"] = pd.cut(df["Amount_INR"], bins=bins, labels=labels, right=True)

    band_summary = df.groupby("Amount_Band", observed=False).agg(
        Transaction_Count=("Transaction_ID", "count"),
        Total_Value_INR=("Amount_INR", "sum")
    )
    band_summary["Volume_Share_Pct"] = (band_summary["Transaction_Count"] / len(df)) * 100

    print("=== FR-12: Transaction Amount Bands ===")
    print(band_summary)

    # FR-13: NumPy Boolean indexing to isolate transactions above INR 1,000
    high_value_mask = amounts_np > threshold
    high_value_amounts = amounts_np[high_value_mask]

    count_above = len(high_value_amounts)
    pct_volume_above = (count_above / len(amounts_np)) * 100
    value_above = np.sum(high_value_amounts)
    pct_value_above = (value_above / np.sum(amounts_np)) * 100

    print(f"\n=== FR-13: NumPy Boolean Indexing (Amount_INR > INR {threshold:,.0f}) ===")
    print(f"Transactions Above Threshold: {count_above} ({pct_volume_above:.2f}% of volume)")
    print(f"Total Value Above Threshold:  INR {value_above:,.2f} ({pct_value_above:.2f}% of total value)")


segment_bands_and_threshold(df_clean, amounts_np, threshold=1000.0)

=== FR-12: Transaction Amount Bands ===
                    Transaction_Count  Total_Value_INR  Volume_Share_Pct
Amount_Band                                                             
Very Low (<=250)                  387        56,566.14             32.80
Low (250-500)                     336       121,326.44             28.47
Medium (500-1,000)                277       195,303.58             23.47
High (1,000-2,500)                146       215,514.95             12.37
Very High (>2,500)                 34       139,025.72              2.88

=== FR-13: NumPy Boolean Indexing (Amount_INR > INR 1,000) ===
Transactions Above Threshold: 180 (15.25% of volume)
Total Value Above Threshold:  INR 354,540.67 (48.72% of total value)


## FR-14: Five Cautious Operational Observations & Recommendations

> **Limitations & Disclaimer:** Findings apply strictly to the 1,180 complete records remaining after removing 20 missing-amount rows from the expanded 1,200-entry synthetic teaching dataset (`10_bank_transactions_1200.csv`). The `Flagged` column is a pre-supplied binary indicator and **must not** be used to claim or prove fraud. Observed patterns reflect descriptive associations only; correlation does not prove causation.

1. **Tiered High-Value Review:** Only **15.25%** of complete transactions (180 records > INR 1,000) account for **48.72%** of total monetary value (INR 354,540.67), and none of the top 10 largest transactions carry a flag.  
   *Recommendation:* Evaluate stepped verification checks for high-value transfers while keeping routine low-value payments (61.27% <= INR 500) frictionless.
2. **Digital Scaling & Branch Migration:** **NetBanking** leads both transaction volume (257; 21.78%) and value (INR 170,575.90), while **Branch** remains a close second in volume (252; 21.36%).  
   *Recommendation:* Maintain high digital uptime for NetBanking and encourage migration of routine small-ticket Branch transactions to NetBanking or UPI.
3. **Careful Validation of Channel Flag Rates:** Among complete records, **50 transactions (4.24%)** are flagged, with higher observed percentages in **Branch** (5.56%; 14/252) and **UPI** (4.70%; 11/234) compared with **Card** (2.94%; 6/204).  
   *Recommendation:* Because channel-level flagged counts remain modest (6–14 records), validate these differences on a larger production sample before altering channel monitoring rules.
4. **Ingestion Schema Validation for Missing Amounts:** Exactly **20 records (1.67%)** had blank `Amount_INR` entries—including 3 flagged transactions—and had to be removed during cleaning.  
   *Recommendation:* Enforce strict non-null validation checks on `Amount_INR` at transaction-ingestion gateways so incomplete monetary records (especially flagged ones) are never lost or dropped downstream.
5. **Longitudinal Regional & Account Monitoring:** **Kolkata** (INR 169,020.60; 23.23%) and **Mumbai** (INR 168,652.56; 23.17%) together contribute **46.40%** of total value compared with **Chennai** (16.01%), while **Student** accounts record strong Credit inflows (INR 140,275.23 Credit vs. INR 116,647.17 Debit) and **Savings** accounts are nearly balanced with slightly higher Debits (INR 115,443.77 vs. INR 114,126.30).  
   *Recommendation:* Track regional ticket sizes and account-type cash flows across multi-year real datasets before adjusting liquidity allocation or product strategies.